# Nehalennia data layer

Harvests each source into a dataframe, then writes the columns RMO and RMA share to
`presentation_layer/data/objecten.json`.

In [1]:
# %pip install pandas openpyxl rdflib

import pandas as pd

import enrich
import links
import source_collectie_zeeland
import source_rma
import source_rmo
from const import CACHE_DIR, DATA_DIR, make_cache_dir

make_cache_dir(CACHE_DIR)

OBJECTEN_PATH = DATA_DIR / "objecten.json"
COLUMNS = [
    "pid",
    "bron",
    "objectnummer",
    "titel",
    "objectnaam",
    "beschrijving",
    "vindplaats",
    "datering_datum_start",
    "datering_datum_eind",
    "gevonden_datum_van",
    "gevonden_datum_tot",
    "image_url",
    "inscripties_text",
    "inscripties",
    "edcs_vertaling",
    "period_ids",
    "linked_objecten",
]


def write_json(path, frames, columns=COLUMNS):
    """COLUMNS in that order, whatever a frame is missing comes out null"""
    objecten = pd.concat([frame.reindex(columns=columns) for frame in frames],
                         ignore_index=True)

    # escape float nan
    for column in columns:
        owners = [frame[column] for frame in frames if column in frame.columns]
        if owners and all(pd.api.types.is_integer_dtype(owner) for owner in owners):
            objecten[column] = objecten[column].astype("Int64")

    return write_records(path, objecten)


def write_records(path, frame):
    """One JSON array, one object per row"""
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_json(path, orient="records", indent=2, force_ascii=False)
    return frame

## RMO

In [2]:
df_rmo, df_rmo_images = source_rmo.fetch(cache_dir=CACHE_DIR)
df_rmo.head()

nehalennia: 276 records
colijnsplaat: 386 records
411 unique records
reading 411 object pages for verwerving, cached ones are reused
    25/411     0s elapsed, ~0s to go
    50/411     0s elapsed, ~0s to go
    75/411     0s elapsed, ~0s to go
   100/411     0s elapsed, ~0s to go
   125/411     0s elapsed, ~0s to go
   150/411     0s elapsed, ~0s to go
   175/411     0s elapsed, ~0s to go
   200/411     0s elapsed, ~0s to go
   225/411     0s elapsed, ~0s to go
   250/411     0s elapsed, ~0s to go
   275/411     0s elapsed, ~0s to go
   300/411     0s elapsed, ~0s to go
   325/411     0s elapsed, ~0s to go
   350/411     0s elapsed, ~0s to go
   375/411     0s elapsed, ~0s to go
   400/411     0s elapsed, ~0s to go
   411/411     0s elapsed, ~0s to go
116 of 159 inscriptions matched at >=75%, 1 candidates each on average, worst 3 of 137


,pid,objectnummer,titel,objectnaam,periode,datering_datum_start,datering_datum_eind,vindplaats,inscripties_text,beschrijving,bron,image_url,verwerving,gevonden_datum_van,gevonden_datum_tot,edcs_id,inscripties,edcs_score,edcs_vertaling
0,https://hdl.handle.net/21.12126/137816,137816,Nehalennia-altaar,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DE[AE] NEHALENNI[AE] / [.] VAL MAR[......] / N...,Altaar met nis. Vooraan rechtsonder is een hoe...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019732,opgraving RMO 1970,1970,1970,EDCS-09401505,https://edcs.hist.uzh.ch/monument/09401505,0.872,Aan de godin Nehalennia. [...] Valerius Mar[.....
1,https://hdl.handle.net/21.12126/137817,137817,Nehalennia-altaar,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DEAE . NEHALEN / NIAE . C . IVLIVS / PRIMITIVS...,"Altaar met nis. Kalksteen. Nehalennia zittend,...",Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019527,opgraving RMO 1970,1970,1970,EDCS-09401493,https://edcs.hist.uzh.ch/monument/09401493,1.000,Aan de godin Nehalennia. Gaius Iulius Primitiv...
2,https://hdl.handle.net/21.12126/137818,137818,Nehalennia-altaar,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",D . NEHALE / [N]IAE . PRIMAN / VS VRSVL[I] / P...,Altaar. Kalksteen. Rechter- en linkerkant: boo...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019423,opgraving RMO 1970,1970,1970,EDCS-09401502,https://edcs.hist.uzh.ch/monument/09401502,0.941,"Aan de godin Nehalennia. Iramanus, [zoon] van ..."
3,https://hdl.handle.net/21.12126/137819,137819,Nehalennia-altaar,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",NEHA[L]ENI[.] / [.]IMA[.....] / [---] / V [---],Altaar met nis. Kalksteen. Afwijkend type in z...,Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/015782,opgraving RMO 1970,1970,1970,EDCS-23400585 | EDCS-23400676 | EDCS-23400660,https://edcs.hist.uzh.ch/monument/23400585 | h...,0.828 | 0.783 | 0.783,Aan Nehalennia. [...]ima[...] [6 tekens verlor...
4,https://hdl.handle.net/21.12126/138005,138005,Nehalennia-altaar,altaar,Romeins 150-250 n.Chr.,150,250,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",DEAE / NEHALENIAE/ M . EXGINGIVS / AGRICOLA / ...,"Altaar met nis, kalksteen. Nehalennia zittend....",Rijksmuseum van Oudheden,https://www.rmo.nl/imageproxy/tiff/019737,opgraving RMO 1970,1970,1970,EDCS-09401487,https://edcs.hist.uzh.ch/monument/09401487,1.000,Aan de godin Nehalennia. Marcus Exgingius Agri...


## RMA

In [3]:
df_rma = source_rma.fetch(cache_dir=CACHE_DIR)
df_rma[["objectnummer", "titel", "image_url", "edcs_vertaling"]].head()

48 rows in source/rma_nehalennia.csv
   25/48  0s elapsed
   48/48  0s elapsed
48 image urls, 45 translations


,objectnummer,titel,image_url,edcs_vertaling
0,RP-P-OB-26.650,Altaar van de Romeinse godin Nehalennia,https://iiif.micr.io/kneUt/full/max/0/default.jpg,Aan de godin Nehalennia. Ianuarinius Ambacthiu...
1,RP-P-OB-26.651,,https://iiif.micr.io/UnsXr/full/max/0/default.jpg,"Aan de godin Nehalennia. Flettius, zoon van Ge..."
2,RP-P-OB-50.029,Nehalennia-altaar\nOudheden gevonden op het st...,https://iiif.micr.io/TVQKh/full/max/0/default.jpg,"Aan de godin Nehalennia. Flettius, zoon van Ge..."
3,RP-P-OB-50.030,Nehalennia-altaar\nOudheden gevonden op het st...,https://iiif.micr.io/fSKXb/full/max/0/default.jpg,Aan de godin Nehalennia. Sextus Nertomarius Ne...
4,RP-P-OB-50.031,Nehalennia-altaar\nOudheden gevonden op het st...,https://iiif.micr.io/xIEWo/full/max/0/default.jpg,Aan de godin Nehalennia. [naam onleesbaar] hee...


## Collectie Zeeland

Streams the dump on every run (no cache)

In [4]:
df_zeeland = source_collectie_zeeland.fetch(ttl_path=CACHE_DIR / "zeeland_nehalennia.ttl")
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger"]]

318042 triples in dump
wrote 248 triples to cache/zeeland_nehalennia.ttl


,objectnummer,titel,objecttype,materiaal,vervaardiger
0,MDC0139,Nehalennia,beelden (CHT),,"Metsers, Guido (RKDartists)"
1,MDC0225,Nehalennia,objecten (CHT),,Ineke de Brouwer
2,MCA00408,Votiefaltaar (replica) voor de godin Nehalennia,votiefaltaar,composiet,
3,MCA00113,Randfragment van een groot voorraadvat of dolium,dolium aardewerk:dolium groot formaat (ABR),aardewerk,


## Export

In [5]:
df_rmo = enrich.periods(df_rmo)
df_rma = enrich.periods(df_rma)

df_rmo, df_rma = links.attach([df_rmo, df_rma])

objecten = write_json(OBJECTEN_PATH, [df_rmo, df_rma])
objecten.head()

306 of 411 objects placed in a period
48 of 48 objects placed in a period
55 groups over 118 objects {'inscriptie': 51, 'onderzoek': 4}


,pid,bron,objectnummer,titel,objectnaam,beschrijving,vindplaats,datering_datum_start,datering_datum_eind,gevonden_datum_van,gevonden_datum_tot,image_url,inscripties_text,inscripties,edcs_vertaling,period_ids,linked_objecten
0,https://hdl.handle.net/21.12126/137816,Rijksmuseum van Oudheden,137816,Nehalennia-altaar,altaar,Altaar met nis. Vooraan rechtsonder is een hoe...,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",150,250,1970,1970,https://www.rmo.nl/imageproxy/tiff/019732,DE[AE] NEHALENNI[AE] / [.] VAL MAR[......] / N...,https://edcs.hist.uzh.ch/monument/09401505,Aan de godin Nehalennia. [...] Valerius Mar[.....,"[200, 20ste]",[]
1,https://hdl.handle.net/21.12126/137817,Rijksmuseum van Oudheden,137817,Nehalennia-altaar,altaar,"Altaar met nis. Kalksteen. Nehalennia zittend,...","Nederland, Zeeland, Noord-Beveland, Colijnspla...",150,250,1970,1970,https://www.rmo.nl/imageproxy/tiff/019527,DEAE . NEHALEN / NIAE . C . IVLIVS / PRIMITIVS...,https://edcs.hist.uzh.ch/monument/09401493,Aan de godin Nehalennia. Gaius Iulius Primitiv...,"[200, 20ste]",[]
2,https://hdl.handle.net/21.12126/137818,Rijksmuseum van Oudheden,137818,Nehalennia-altaar,altaar,Altaar. Kalksteen. Rechter- en linkerkant: boo...,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",150,250,1970,1970,https://www.rmo.nl/imageproxy/tiff/019423,D . NEHALE / [N]IAE . PRIMAN / VS VRSVL[I] / P...,https://edcs.hist.uzh.ch/monument/09401502,"Aan de godin Nehalennia. Iramanus, [zoon] van ...","[200, 20ste]",[]
3,https://hdl.handle.net/21.12126/137819,Rijksmuseum van Oudheden,137819,Nehalennia-altaar,altaar,Altaar met nis. Kalksteen. Afwijkend type in z...,"Nederland, Zeeland, Noord-Beveland, Colijnspla...",150,250,1970,1970,https://www.rmo.nl/imageproxy/tiff/015782,NEHA[L]ENI[.] / [.]IMA[.....] / [---] / V [---],https://edcs.hist.uzh.ch/monument/23400585 | h...,Aan Nehalennia. [...]ima[...] [6 tekens verlor...,"[200, 20ste]",[{'pid': 'https://hdl.handle.net/21.12126/1590...
4,https://hdl.handle.net/21.12126/138005,Rijksmuseum van Oudheden,138005,Nehalennia-altaar,altaar,"Altaar met nis, kalksteen. Nehalennia zittend....","Nederland, Zeeland, Noord-Beveland, Colijnspla...",150,250,1970,1970,https://www.rmo.nl/imageproxy/tiff/019737,DEAE / NEHALENIAE/ M . EXGINGIVS / AGRICOLA / ...,https://edcs.hist.uzh.ch/monument/09401487,Aan de godin Nehalennia. Marcus Exgingius Agri...,"[200, 20ste]",[]
